# 📄 PaperMindAI: Capstone Project
**Subtitle**: Local Retrieval-Augmented Generation (RAG) System for Complex PDF Research Papers

This notebook documents the fully implemented, end-to-end architecture and evaluation of the PaperMindAI project. It runs the entire pipeline from document ingestion to the final generated response with citations.

## 1. Problem Statement
**Business Context**: Researchers, students, and professionals spend countless hours manually reading and parsing lengthy PDF documents to find specific information or answer nuanced questions.
**Objective**: Build an automated, hallucination-free Question Answering (QA) system that accurately retrieves information directly from uploaded documents.
**Use Case**: A system where users upload research papers, ask natural language questions, and receive precise answers backed by inline citations and original source page images.

## 2. Setup & Imports
We import the custom modules built for this project: `loader`, `chunking`, `embeddings`, `vectorstore`, `retriever`, and `rag_pipeline`.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables (like GEMINI_API_KEY)
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY", "")

# Import custom project modules
from loader import load_documents_from_paths
from chunking import split_documents
from embeddings import get_huggingface_embeddings, get_gemini_embeddings
from vectorstore import create_and_save_vectorstore
from retriever import get_retriever
from rag_pipeline import generate_answer

c:\Users\LOCHANA\Downloads\PaperMindAI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\LOCHANA\Downloads\PaperMindAI\venv\Scripts\python.exe
3.10.8 (tags/v3.10.8:aaaf517, Oct 11 2022, 16:50:30) [MSC v.1933 64 bit (AMD64)]


## 3. Data Ingestion
**Dataset Details**: Any unstructured PDF files. For this experiment, we use an academic research paper located in `pdf_uploads/`.
**Loader Used**: PyMuPDF (`fitz` and `pymupdf4llm`). We chose this because standard PyPDF often mangles tables, whereas PyMuPDF natively extracts pages into structured Markdown and supports image extraction for multimodal RAG.

In [3]:
# Load a sample PDF from our uploads directory
sample_pdf = "pdf_uploads/RAG.pdf"

print(f"Loading document: {sample_pdf}")
docs = load_documents_from_paths([sample_pdf], api_key=api_key, extract_images=False)
print(f"Loaded {len(docs)} pages.")

Loading document: pdf_uploads/RAG.pdf
Loaded 36 pages (text only) from pdf_uploads/RAG.pdf
Loaded 36 pages.


## 4. Text Chunking
**Chunking Strategies Compared**:
- **Strategy A**: Chunk Size 1000, Overlap 200. Good for general dense text and definitions.
- **Strategy B**: Chunk Size 1500, Overlap 300. Good for complex academic papers, tables, and code.

**Chosen Strategy**: We use **Strategy B** because the larger chunk size keeps complex code elements, tables, and long-form paragraphs together, preventing loss of context. The 300-character overlap ensures that critical sentences spanning across chunks are not broken in half.

In [4]:
# Split the loaded document into manageable chunks
chunks = split_documents(docs, strategy="B")
print(f"Document split into {len(chunks)} chunks.")
print(f"Sample chunk preview: {chunks[0].page_content[:150]}...")

Split 36 documents into 69 chunks using Strategy B (1500/300).
Document split into 69 chunks.
Sample chunk preview: # **Developing Retrieval Augmented Generation (RAG) based LLM Systems from PDFs: An Experience Report** 

**Ayman Asad Khan** Tampere University ayman...


## 5. Embeddings
**Models Compared**:
- **Open Source**: HuggingFace (`BAAI/bge-small-en`). Fast, runs entirely locally, ensures data privacy, zero API costs.
- **Commercial**: Google Gemini (`gemini-embedding-001`). Offers state-of-the-art semantic understanding but requires API access.

**Final Choice**: We support both, but for this notebook, we use the local **HuggingFace** embeddings to demonstrate a fast, privacy-preserving semantic representation.

In [5]:
# Initialize the HuggingFace embedding model
print("Initializing HuggingFace Embeddings...")
embeddings = get_huggingface_embeddings()

Initializing HuggingFace Embeddings...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1126.85it/s]


## 6. Vector DB & Retrieval
**Vector Database Used**: **FAISS** (Facebook AI Similarity Search). Chosen for its blazing fast, in-memory execution, perfect for local setups.

**Retrieval Strategies Compared**:
- **Similarity**: Standard Cosine Similarity. Quickest for exact snippets.
- **MMR (Max Marginal Relevance)**: Relevance + Diversity. Best for summaries spanning multiple sections.
- **Hybrid Search**: Combines BM25 (keyword search) with FAISS (semantic search).
- **Reranker**: Uses an MMR base retriever wrapped in a HuggingFace Cross-Encoder to re-score chunks.

**Final Selection**: We will demonstrate the **Reranker** strategy to maximize the precision of the final context window.

In [6]:
# Create the FAISS Vector Store
print("Building FAISS Vector Store...")
vectorstore = create_and_save_vectorstore(chunks, embeddings)

# Configure the Retriever
# Using the advanced 'reranker' strategy (MMR + Cross-Encoder) and fetching top 3 chunks
retriever = get_retriever(vectorstore, strategy="reranker", top_k=3)

Building FAISS Vector Store...
Creating FAISS index with 69 chunks...
Processing batch 1/7...
Processing batch 2/7...
Processing batch 3/7...
Processing batch 4/7...
Processing batch 5/7...
Processing batch 6/7...
Processing batch 7/7...
FAISS index saved to faiss_index


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1674.04it/s]


Configured Reranker retriever (k=3)


## 7. RAG Pipeline
**Chain Construction**: Built using LangChain Expression Language (LCEL).
**Prompt Template**: Extremely strict instructions mandating the LLM to *only* use provided context, output "I don't know" if the answer isn't present (zero hallucinations), and highlight "Note:" segments.
**LLM Used**: Google Gemini (`gemini-3.1-flash-lite-preview`).

In [7]:
# Define a test query
query = "What is the main contribution or objective discussed in this paper?"

print(f"Querying the LLM: '{query}'\n")

# Run the generation pipeline
answer, retrieved_docs = generate_answer(
    query=query,
    retriever=retriever,
    api_key=api_key,
    model_name="gemini-3.1-flash-lite-preview",
    chat_history=[]
)

print("### GENERATED ANSWER ###")
print(answer)

Querying the LLM: 'What is the main contribution or objective discussed in this paper?'

### GENERATED ANSWER ###
I don't know.


## 8. Results & Source Citations
The final step is to verify the results. We ensure the LLM returns exactly the **Top 3 supporting citations**, including the exact paper title/filename and page number.

In [8]:
# Display the top 3 citations
print("### TOP 3 SUPPORTING CITATIONS ###\n")

for idx, doc in enumerate(retrieved_docs[:3]):
    # Extract metadata safely
    source = doc.metadata.get('source', 'Unknown Document')
    filename = os.path.basename(source) if source != 'Unknown Document' else 'Unknown Document'
    page = doc.metadata.get('page', 'Unknown Page')
    
    print(f"Citation [{idx + 1}]:")
    print(f"- Paper Title / File: {filename}")
    print(f"- Page Number: {page}")
    print(f"- Snippet Preview: {doc.page_content[:200].replace(chr(10), ' ')}...\n")


### TOP 3 SUPPORTING CITATIONS ###

Citation [1]:
- Paper Title / File: RAG.pdf
- Page Number: 16
- Snippet Preview: ```  ### 2. **Understanding the Problem Domain and Data Requirements**   To develop an effective solution for managing and retrieving information, it’s important to understand the problem domain and i...

Citation [2]:
- Paper Title / File: RAG.pdf
- Page Number: 31
- Snippet Preview: in machine learning, natural language processing (NLP), and using tools for Retrieval Augmented Generation (RAG). The participants although had familiarity with Python language and OpenAI models.   ##...

Citation [3]:
- Paper Title / File: RAG.pdf
- Page Number: 28
- Snippet Preview: ``` Youareanexpertassistantwithaccesstothe followingcontextextractedfromdocuments.Your jobistoanswertheuser ’squestionasaccurately aspossible ,usingthecontextbelow. Context: {context} Giventhisinforma...

